In [2]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_linear_schedule_with_warmup

c:\Users\acer\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
train_text = np.load("C:/EmoBase/features/train_text.npy")
train_audio = np.load("C:/EmoBase/features/train_audio.npy")
train_face= np.load("C:/EmoBase/features/train_face.npy")

train_features = np.concatenate((train_text, train_audio, train_face), axis=1)

dev_text= np.load("C:/EmoBase/features/dev_text.npy")
dev_audio= np.load("C:/EmoBase/features/dev_audio.npy")
dev_face= np.load("C:/EmoBase/features/dev_face.npy")

dev_features = np.concatenate((dev_text, dev_audio, dev_face), axis=1)

test_text= np.load("C:/EmoBase/features/test_text.npy")
test_audio= np.load("C:/EmoBase/features/test_audio.npy")
test_face= np.load("C:/EmoBase/features/test_face.npy")

test_features = np.concatenate((test_text, test_audio, test_face), axis=1) 

print(train_features.shape,test_features.shape,dev_features.shape)

(9989, 3584) (2610, 3584) (1109, 3584)


In [14]:
df_train=pd.read_csv("C:/MELD.Raw/train/train_sent_emo.csv")
df_dev=pd.read_csv("C:/MELD.Raw/dev/dev_sent_emo.csv")
df_test=pd.read_csv("C:/MELD.Raw/test/test_sent_emo.csv")

label_map= {'anger': 0, 'disgust': 1, 'fear': 2, 'joy': 3, 'neutral': 4, 'sadness': 5, 'surprise': 6}

df_train['label'] = df_train['Emotion'].map(label_map)
df_test['label'] = df_test['Emotion'].map(label_map)
df_dev['label'] = df_dev['Emotion'].map(label_map)

In [15]:
df_train = df_train.sort_values(['Dialogue_ID', 'Utterance_ID'])
train_features = train_features[df_train.index]
df_train = df_train.reset_index(drop=True)

df_dev = df_dev.sort_values(['Dialogue_ID', 'Utterance_ID'])
dev_features = dev_features[df_dev.index]
df_dev = df_dev.reset_index(drop=True)

df_test = df_test.sort_values(['Dialogue_ID', 'Utterance_ID'])
test_features = test_features[df_test.index]
df_test = df_test.reset_index(drop=True)

In [16]:
df_train[['Dialogue_ID', 'Utterance_ID', 'label']].head(10)

,Dialogue_ID,Utterance_ID,label
0,0,0,4
1,0,1,4
2,0,2,4
3,0,3,4
4,0,4,6
5,0,5,4
6,0,6,4
7,0,7,4
8,0,8,4
9,0,9,4


In [ ]:
def build_emotion_history(df, window_size=5):
    histories = []
    dialogue_emotions = {}

    for index, row in df.iterrows():
        dialogue = row['Dialogue_ID']
        label = row['label']

        if dialogue not in dialogue_emotions:
            dialogue_emotions[dialogue] = []

        current_history = dialogue_emotions[dialogue][-window_size:]
        padding_size = window_size - len(current_history)
        padding = [0] * padding_size
        history = padding + current_history
        histories.append(history)
        dialogue_emotions[dialogue].append(int(label))

    return np.array(histories)

train_history = build_emotion_history(df_train)
dev_history = build_emotion_history(df_dev)
test_history = build_emotion_history(df_test)

(9989, 5)
(1109, 5)
(2610, 5)
